<center>
<a href="https://www.umontpellier.fr/"><img src="https://www.umontpellier.fr/wp-content/uploads/2022/10/logo_um_2022_rouge_rvb.svg" width="200"/></a>&nbsp;&nbsp;
<a href="https://economie.edu.umontpellier.fr/"><img src="https://economie.edu.umontpellier.fr/files/2014/12/economie_rvb_2015-300x137.png" width="160"/></a>
</center>

<div align="center">

#  Phase 1 — Nettoyage et Validation de la Base `dossier.csv`

| Nom et Prénom | Rôle |
|---|---|
| Randriamisaina Tsiory-Fanomezana | Membre de l'équipe |
| SHIRALI POUR Amir | Membre de l'équipe |

</div>

---
## Objectif de cette phase

Ce notebook constitue la **première phase** du pipeline de traitement des données.  
Il applique, dans l'ordre, toutes les corrections décrites dans [`docs/phase1_dossier.md`](../docs/phase1_dossier.md).

Pour chaque anomalie, la démarche est systématique :
1. **Constat** — quantification du problème
2. **Décision** — choix documenté avec justification métier/statistique  
3. **Action** — code de correction
4. **Vérification** — preuve que la correction est effective


---
## Section 0 — Initialisation

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# --- Localisation de la racine du projet
def localiser_racine_du_projet():
    """Remonte l'arborescence jusqu'à trouver la racine du projet (présence de .git ou requirements.txt)."""
    repertoire_courant = Path.cwd()
    while True:
        if any((repertoire_courant / m).exists() for m in ['.git', 'requirements.txt']):
            break
        if repertoire_courant.parent == repertoire_courant:
            break
        repertoire_courant = repertoire_courant.parent
    if Path.cwd() != repertoire_courant:
        os.chdir(repertoire_courant)
    return repertoire_courant.resolve()

REPERTOIRE_RACINE = localiser_racine_du_projet()
sys.path.insert(0, str(REPERTOIRE_RACINE / 'src'))

from utils.dataframe_styler import style_duplicates
print(f" Racine du projet : {REPERTOIRE_RACINE}")

In [ ]:
# --- Chargement de la base brute
dossier_original = pd.read_csv('data/cleaned_dossier.csv', encoding='utf-8', sep=',', quotechar='"')

# Copie de travail — on ne touche JAMAIS à dossier_original
dossier = dossier_original.copy()

print(f" Dimensions chargées : {dossier.shape[0]:,} lignes × {dossier.shape[1]} colonnes")
print(f" Identifiants uniques : {dossier['Numero_dossier_ID'].nunique():,}")
print(f" Identifiants dupliqués : {dossier['Numero_dossier_ID'].duplicated().sum():,}")

---
## Section 1 — Vue d'ensemble de la qualité des données

### 1.1 Types de données et valeurs non-nulles

In [ ]:
dossier.info()

In [ ]:
def compter_anomalies_par_colonne(dataframe):
    """
    Calcule, pour chaque colonne, le nombre de NaN, de '???', et le total.
    Retourne un DataFrame trié par anomalies décroissantes.
    """
    resultats = []
    for colonne in dataframe.columns:
        n_nan            = dataframe[colonne].isna().sum()
        n_interrogation  = (dataframe[colonne].astype(str).str.strip() == '???').sum()
        total_anomalies  = n_nan + n_interrogation
        pourcentage      = round(total_anomalies / len(dataframe) * 100, 2)
        resultats.append({
            'Colonne'        : colonne,
            'NaN'            : n_nan,
            "???"            : n_interrogation,
            'Total anomalies': total_anomalies,
            '% du total'     : pourcentage
        })
    return (pd.DataFrame(resultats)
              .sort_values('Total anomalies', ascending=False)
              .reset_index(drop=True))

tableau_anomalies_initial = compter_anomalies_par_colonne(dossier)
print("Tableau récapitulatif des anomalies (colonnes avec au moins 1 problème) :")
tableau_anomalies_initial[tableau_anomalies_initial['Total anomalies'] > 0]

---
## Section 2 — Anomalie A1 : Doublons sur `Numero_dossier_ID`

### Constat
La base contient **101 234 lignes** mais seulement **100 000 identifiants uniques**,  
soit **1 234 lignes dupliquées**.

### Décision : Suppression — `keep='first'`
> `Numero_dossier_ID` est une **clé primaire** : chaque dossier doit apparaître exactement une fois.  
> Sans information supplémentaire permettant de distinguer la "bonne" occurrence,  
> on retient la **première** (convention standard en gestion de bases de données).  
> Cette approche préserve les 100 000 dossiers distincts sans perte d'information métier.

### 2.1 Identification et visualisation des doublons

In [ ]:
masque_toutes_lignes_doublonnees = dossier.duplicated(subset=['Numero_dossier_ID'], keep=False)
lignes_doublonnees = dossier[masque_toutes_lignes_doublonnees].sort_values('Numero_dossier_ID')

print(f"Lignes impliquées dans des doublons : {masque_toutes_lignes_doublonnees.sum():,}")
print(f"Nombre d'IDs concernés              : {lignes_doublonnees['Numero_dossier_ID'].nunique():,}")
print()
print("Aperçu des 10 premières lignes dupliquées (groupes colorés par ID) :")
lignes_doublonnees.head(10).style_duplicates(id_column='Numero_dossier_ID')

### 2.2 Suppression et vérification

In [ ]:
nombre_lignes_avant_deduplication = len(dossier)

dossier.drop_duplicates(subset=['Numero_dossier_ID'], keep='first', inplace=True)
dossier.reset_index(drop=True, inplace=True)

nombre_lignes_apres_deduplication = len(dossier)
lignes_supprimees = nombre_lignes_avant_deduplication - nombre_lignes_apres_deduplication

print(f" {lignes_supprimees:,} lignes dupliquées supprimées")
print(f" Taille résultante : {nombre_lignes_apres_deduplication:,} lignes")

# Vérification formelle
assert dossier['Numero_dossier_ID'].duplicated().sum() == 0, " Des doublons subsistent !"
print(" Vérification : 0 doublon restant ")

---
## Section 3 — Anomalies A2 & A3 : Formats et plages de dates

### 3.1 Anomalie A2 — Format français `DD/MM/YYYY` dans `date.ouverture` (10 lignes)

**Décision : Conversion vers le format ISO `YYYY/MM/DD`**  
> La date réelle est récupérable sans ambiguïté par `pd.to_datetime(..., dayfirst=True)`.  
> Le format ISO permet des comparaisons chronologiques directes en chaîne de caractères.

In [ ]:
def convertir_date_vers_iso(valeur_date_en_chaine):
    """Convertit n'importe quel format de date parseable vers YYYY/MM/DD. Retourne la valeur originale si échec."""
    try:
        date_parsee = pd.to_datetime(valeur_date_en_chaine, dayfirst=True)
        return date_parsee.strftime('%Y/%m/%d')
    except Exception:
        return valeur_date_en_chaine

for colonne_date in ['date.ouverture', 'date.de.survenance']:
    valeurs_avant_conversion = dossier[colonne_date].copy()
    dossier[colonne_date]    = dossier[colonne_date].apply(convertir_date_vers_iso)
    masque_dates_modifiees   = valeurs_avant_conversion != dossier[colonne_date]
    print(f" {colonne_date} : {masque_dates_modifiees.sum():,} date(s) convertie(s) au format ISO")

### 3.2 Anomalie A3 — Dates hors période 2021–2022

**Décision : Suppression de toutes les lignes hors `[2021/01/01 – 2022/12/31]`**  
> Le glossaire confirme que la base couvre exclusivement 2021–2022.  
> Les 10 dates `2020/01/01` (converties depuis `01/01/2020`) et les 1 000 dates `2023/01/01`  
> (issues de l'année seule `2023`) sont hors-scope et ne peuvent être corrigées sans données externes.  
> La valeur `2023` seule est également inutilisable pour toute analyse temporelle.

In [ ]:
DATE_DEBUT_PERIODE = '2021/01/01'
DATE_FIN_PERIODE   = '2022/12/31'

masque_ouverture_hors_periode  = (
    (dossier['date.ouverture'] < DATE_DEBUT_PERIODE) |
    (dossier['date.ouverture'] > DATE_FIN_PERIODE)
)
masque_survenance_hors_periode = (
    (dossier['date.de.survenance'] < DATE_DEBUT_PERIODE) |
    (dossier['date.de.survenance'] > DATE_FIN_PERIODE)
)
masque_hors_periode_combine = masque_ouverture_hors_periode | masque_survenance_hors_periode

print(f"date.ouverture hors période    : {masque_ouverture_hors_periode.sum():,} ligne(s)")
print(f"date.de.survenance hors période: {masque_survenance_hors_periode.sum():,} ligne(s)")
print(f"Total à supprimer (union)      : {masque_hors_periode_combine.sum():,} ligne(s)")
print()
print("Échantillon des lignes hors période :")
dossier[masque_hors_periode_combine][
    ['Numero_dossier_ID','date.ouverture','date.de.survenance']
].head(10).style_duplicates(highlight_mask=masque_hors_periode_combine)

In [ ]:
nombre_lignes_avant_suppression_dates = len(dossier)

dossier.drop(index=dossier[masque_hors_periode_combine].index, inplace=True)
dossier.reset_index(drop=True, inplace=True)

nombre_lignes_apres_suppression_dates = len(dossier)
print(f" {nombre_lignes_avant_suppression_dates - nombre_lignes_apres_suppression_dates:,} ligne(s) hors période supprimée(s)")
print(f" Taille résultante : {nombre_lignes_apres_suppression_dates:,} lignes")

# Vérification
n_hors_periode_restants = (
    (dossier['date.ouverture'] < DATE_DEBUT_PERIODE) | (dossier['date.ouverture'] > DATE_FIN_PERIODE) |
    (dossier['date.de.survenance'] < DATE_DEBUT_PERIODE) | (dossier['date.de.survenance'] > DATE_FIN_PERIODE)
).sum()
assert n_hors_periode_restants == 0, f" {n_hors_periode_restants} lignes hors période subsistent !"
print(" Vérification : 0 date hors période restante ")

---
## Section 4 — Anomalie A4 : Cohérence temporelle survenance / ouverture

### Constat
Un sinistre doit survenir **avant ou le jour même** de l'ouverture du dossier.  
Toute date de survenance postérieure à l'ouverture est une incohérence.

### Décision : Correction par substitution `date.de.survenance ← date.ouverture`
> On ne supprime pas la ligne car toutes ses autres informations sont exploitables.  
> On choisit la valeur la plus conservative : la date d'ouverture, qui est la borne haute certaine.

In [ ]:
masque_survenance_superieure_a_ouverture = (
    dossier['date.de.survenance'] > dossier['date.ouverture']
)

print(f"Lignes avec survenance > ouverture : {masque_survenance_superieure_a_ouverture.sum():,}")

if masque_survenance_superieure_a_ouverture.sum() > 0:
    print()
    print("Aperçu avant correction :")
    dossier[masque_survenance_superieure_a_ouverture][
        ['Numero_dossier_ID','date.ouverture','date.de.survenance']
    ].head(5).style_duplicates(
        highlight_mask=masque_survenance_superieure_a_ouverture
    )